# Week 4 · Try It 2: Flag Abnormal Flights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%204/anomaly_flight_logs.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

Each flight log is reduced to two numbers: the motor vibration (g RMS) and the battery temperature after landing (°C). The flights are simulated: 300 normal flights for training, and a validation and a test set of 200 normal flights plus 10 faulty ones each (labels: 1 = fault).

The method is the one in the lecture: fit one Gaussian per feature on **normal** flights only, multiply the densities, and flag a flight if p(**x**) < τ. τ is chosen on the validation set by the best F₁, then applied **once** to the test set.

**Questions** (the same as on the slide)
- (a) What are the fitted μ and σ?
- (b) Which τ wins, and with what F₁?
- (c) Which test faults are missed? Why?
- (d) Refit with the faults mixed into training. What changes?

## Part 1 · The data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2027)
MU, SD = (1.0, 42.0), (0.14, 1.4)      # vibration (g RMS), battery temperature (deg C)

def normal(n):
    return np.column_stack([rng.normal(MU[0], SD[0], n), rng.normal(MU[1], SD[1], n)])

def faults():
    loose_prop = np.column_stack([rng.uniform(1.45, 1.9, 3), rng.normal(MU[1], SD[1], 3)])
    hot_battery = np.column_stack([rng.normal(MU[0], SD[0], 3), rng.uniform(45.5, 49.0, 3)])
    both_mild = np.column_stack([rng.uniform(1.25, 1.35, 2), rng.uniform(44.0, 45.0, 2)])
    dropout = np.column_stack([rng.uniform(0.3, 0.5, 2), rng.normal(MU[1], SD[1], 2)])
    return np.vstack([loose_prop, hot_battery, both_mild, dropout])

KINDS = ["loose propeller"] * 3 + ["hot battery"] * 3 + ["both a little high"] * 2 + ["vibration dropout"] * 2

X_train = normal(300)                                   # normal flights only
X_val = np.vstack([normal(200), faults()])
y_val = np.r_[np.zeros(200, int), np.ones(10, int)]
X_test = np.vstack([normal(200), faults()])
y_test = np.r_[np.zeros(200, int), np.ones(10, int)]
print("train", X_train.shape, " validation", X_val.shape, " test", X_test.shape)

## Part 2 · Fit the Gaussians (maximum likelihood: divide by m)

In [ ]:
mu = X_train.mean(axis=0)
var = X_train.var(axis=0)               # np.var divides by m by default
sd = np.sqrt(var)
print(f"mu    = ({mu[0]:.3f}, {mu[1]:.3f})")
print(f"sigma = ({sd[0]:.3f}, {sd[1]:.3f})")

def density(X, mu, var):
    return np.prod(np.exp(-(X - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var), axis=1)

a, b = np.meshgrid(np.linspace(0.2, 2.0, 200), np.linspace(36, 50, 200))
pp = density(np.column_stack([a.ravel(), b.ravel()]), mu, var).reshape(a.shape)
plt.contour(a, b, pp, levels=[1e-4, 1e-3, 1e-2, 1e-1], colors="0.6")
plt.scatter(*X_train.T, s=8, label="training (normal)")
plt.scatter(*X_val[y_val == 1].T, marker="^", c="r", label="validation faults")
plt.xlabel("motor vibration (g)"); plt.ylabel("battery temperature (°C)"); plt.legend(); plt.show()

## Part 3 · Choose τ on the validation set

In [ ]:
def prf(flag, y):
    tp = int(np.sum(flag & (y == 1))); fp = int(np.sum(flag & (y == 0))); fn = int(np.sum(~flag & (y == 1)))
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn)
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F1

p_val = density(X_val, mu, var)
q = np.unique(p_val)
taus = np.sqrt(q[:-1] * q[1:])          # cut half-way (geometric mean) between neighbouring densities
F1s = [prf(p_val < t, y_val)[2] for t in taus]
tau = taus[int(np.argmax(F1s))]
P, R, F1 = prf(p_val < tau, y_val)
print(f"best tau = {tau:.2g}:  validation precision {P:.2f}, recall {R:.2f}, F1 {F1:.3f}")

plt.semilogx(taus, F1s); plt.axvline(tau, color="r", ls="--")
plt.xlabel("tau (log scale)"); plt.ylabel("F1 on validation"); plt.show()

## Part 4 · Apply τ once to the test set

In [ ]:
p_test = density(X_test, mu, var)
flag = p_test < tau
P, R, F1 = prf(flag, y_test)
print(f"test: precision {P:.2f}, recall {R:.2f}, F1 {F1:.3f}")

z = (X_test - mu) / np.sqrt(var)        # how many standard deviations from the mean
for i in np.where((y_test == 1) & ~flag)[0]:
    print(f"missed: {KINDS[i - 200]:18s}  x = ({X_test[i, 0]:.2f} g, {X_test[i, 1]:.1f} °C),  "
          f"z = ({z[i, 0]:.1f}, {z[i, 1]:.1f}),  p = {p_test[i]:.2g}")

## Part 5 · What if faulty flights get into the training data?

In [ ]:
X_dirty = np.vstack([X_train, X_val[y_val == 1]])      # the 10 validation faults slip into training
mu_d, var_d = X_dirty.mean(axis=0), X_dirty.var(axis=0)
P, R, F1 = prf(density(X_test, mu_d, var_d) < tau, y_test)
print(f"sigma with faults in training: ({np.sqrt(var_d[0]):.3f}, {np.sqrt(var_d[1]):.3f})")
print(f"test with the same tau: precision {P:.2f}, recall {R:.2f}, F1 {F1:.3f}")

---
## Check your answers (try first!)

- (a) μ = (1.011, 42.003), σ = (0.147, 1.519): close to the true (1.0, 42.0) and (0.14, 1.4).
- (b) τ = 0.0029, validation F₁ = 0.889.
- (c) Test: precision 1.00, recall 0.80, F₁ = 0.889. The two missed faults are the 'both a little high' flights: the vibration is about 2σ above its mean and the temperature only 1.3σ to 1.6σ, so neither looks rare on its own and the product of the two densities stays above τ. A model with the full covariance (Appendix A) or a third feature could catch them.
- (d) σ grows to (0.166, 1.62), so faulty values look less unusual, and the test recall drops from 0.8 to 0.6. Fit the model on known-good flights only.